In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


In [2]:
DATA_DIR = Path("../../ml/data/model3_salaries/3.processed")
CSV_PATH = DATA_DIR / "salaries_train_v3.csv"

df = pd.read_csv(CSV_PATH)

In [3]:
print("Shape:", df.shape)
display(df.head())

Shape: (5607, 11)


,source_dataset,work_year,role_raw,role_label_salary,seniority_raw,seniority,country,remote_ratio,company_size,employment_type,salary_in_usd
0,ds_salaries,2020,Data Scientist,data_scientist,MI,mid,DE,0,L,FT,79833.0
1,ds_salaries,2020,Machine Learning Scientist,machine_learning_engineer,SE,senior,JP,0,S,FT,260000.0
2,ds_salaries,2020,Big Data Engineer,data_engineer,SE,senior,GB,50,M,FT,109024.0
3,ds_salaries,2020,Product Data Analyst,data_analyst,MI,mid,HN,0,S,FT,20000.0
4,ds_salaries,2020,Machine Learning Engineer,machine_learning_engineer,SE,senior,US,50,L,FT,150000.0


In [4]:
print("Null %:")
display((df.isna().mean() * 100).round(2).sort_values(ascending=False))

print("\nrole_label_salary distribution:")
display(df["role_label_salary"].value_counts())

print("\nSalary stats:")
display(df["salary_in_usd"].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))


Null %:


source_dataset       0.0
work_year            0.0
role_raw             0.0
role_label_salary    0.0
seniority_raw        0.0
seniority            0.0
country              0.0
remote_ratio         0.0
company_size         0.0
employment_type      0.0
salary_in_usd        0.0
dtype: float64


role_label_salary distribution:


data_engineer                1704
data_scientist               1569
data_analyst                  990
machine_learning_engineer     851
research_scientist            259
bi_engineer                   168
other                          58
project_manager                 8
Name: role_label_salary, dtype: int64


Salary stats:


count      5607.000000
mean     144959.360264
std       68374.828450
min        2859.000000
1%        21669.000000
5%        50000.000000
25%       98000.000000
50%      138000.000000
75%      184700.000000
95%      260000.000000
99%      336132.000000
max      750000.000000
Name: salary_in_usd, dtype: float64

In [5]:
TARGET = "salary_in_usd"
df["log_salary"] = np.log1p(df[TARGET])

FEATURES_CATEGORICAL = [
    "role_label_salary",
    "seniority",
    "country",
    "company_size",
    "employment_type",
    # probaremos luego con y sin source_dataset
    "source_dataset",
]

FEATURES_NUMERIC = [
    "work_year",
    "remote_ratio",
]

X = df[FEATURES_CATEGORICAL + FEATURES_NUMERIC].copy()
y = df["log_salary"].copy()

print("X shape:", X.shape, "| y shape:", y.shape)


X shape: (5607, 8) | y shape: (5607,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=df["role_label_salary"]
)

print("Train:", X_train.shape, "Val:", X_val.shape)


Train: (4485, 8) Val: (1122, 8)


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse=False), FEATURES_CATEGORICAL),
        ("num", "passthrough", FEATURES_NUMERIC),
    ],
    remainder="drop",
)

model = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.08,
    max_iter=300,
    random_state=42,
)

pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", model),
    ]
)

pipe


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse=False),
                                                  ['role_label_salary',
                                                   'seniority', 'country',
                                                   'company_size',
                                                   'employment_type',
                                                   'source_dataset']),
                                                 ('num', 'passthrough',
                                                  ['work_year',
                                                   'remote_ratio'])])),
                ('model',
                 HistGradientBoostingRegressor(learning_rate=0.08, max_depth=6,
                                               max_iter=300,
                                               random_state=42))])

In [8]:
from sklearn.metrics import mean_absolute_error

pipe.fit(X_train, y_train)

# predicciones en log
pred_train_log = pipe.predict(X_train)
pred_val_log = pipe.predict(X_val)

# volver a USD
y_train_usd = np.expm1(y_train)
y_val_usd = np.expm1(y_val)

pred_train_usd = np.expm1(pred_train_log)
pred_val_usd = np.expm1(pred_val_log)

mae_train = mean_absolute_error(y_train_usd, pred_train_usd)
mae_val = mean_absolute_error(y_val_usd, pred_val_usd)

print(f"MAE train: ${mae_train:,.0f}")
print(f"MAE val  : ${mae_val:,.0f}")


c:\Users\Fiona A\anaconda3\lib\site-packages\sklearn\preprocessing\_encoders.py:828: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


MAE train: $37,255
MAE val  : $40,019


In [9]:
from sklearn.metrics import mean_absolute_error
import numpy as np

val_eval = X_val.copy()
val_eval["y_true"] = np.expm1(y_val.values)
val_eval["y_pred"] = np.expm1(pipe.predict(X_val))

role_mae = (
    val_eval.groupby("role_label_salary")[["y_true", "y_pred"]]
    .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
    .sort_values()
)

print("MAE by role (USD):")
display(role_mae.round(0))


MAE by role (USD):


role_label_salary
data_analyst                 26585.0
other                        35283.0
data_engineer                40585.0
data_scientist               41531.0
bi_engineer                  42448.0
machine_learning_engineer    42702.0
project_manager              68325.0
research_scientist           68519.0
dtype: float64

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

FEATURES_CATEGORICAL_NO_SRC = [
    "role_label_salary",
    "seniority",
    "country",
    "company_size",
    "employment_type",
]
FEATURES_NUMERIC = ["work_year", "remote_ratio"]

X2 = df[FEATURES_CATEGORICAL_NO_SRC + FEATURES_NUMERIC].copy()
y2 = df["log_salary"].copy()

X2_train, X2_val, y2_train, y2_val = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=df["role_label_salary"]
)

pre2 = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse=False), FEATURES_CATEGORICAL_NO_SRC),
        ("num", "passthrough", FEATURES_NUMERIC),
    ]
)


model2 = HistGradientBoostingRegressor(
    max_depth=6, learning_rate=0.08, max_iter=300, random_state=42
)

pipe2 = Pipeline([("preprocess", pre2), ("model", model2)])
pipe2.fit(X2_train, y2_train)

pred2_val = np.expm1(pipe2.predict(X2_val))
true2_val = np.expm1(y2_val)

mae2_val = mean_absolute_error(true2_val, pred2_val)
print(f"MAE val (NO source_dataset): ${mae2_val:,.0f}")


c:\Users\Fiona A\anaconda3\lib\site-packages\sklearn\preprocessing\_encoders.py:828: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


MAE val (NO source_dataset): $40,020


In [12]:
from sklearn.metrics import mean_absolute_error
import numpy as np

last_year = df["work_year"].max()
train_mask = df["work_year"] < last_year
val_mask = df["work_year"] == last_year

print("last_year:", last_year)
print("train rows:", train_mask.sum(), "| val rows:", val_mask.sum())

X_time_train = X2[train_mask].copy()
y_time_train = y2[train_mask].copy()

X_time_val = X2[val_mask].copy()
y_time_val = y2[val_mask].copy()

pipe2_time = Pipeline([("preprocess", pre2), ("model", model2)])
pipe2_time.fit(X_time_train, y_time_train)

pred_time_usd = np.expm1(pipe2_time.predict(X_time_val))
true_time_usd = np.expm1(y_time_val)

mae_time = mean_absolute_error(true_time_usd, pred_time_usd)
print(f"Temporal MAE val (year={last_year}): ${mae_time:,.0f}")


last_year: 2024
train rows: 4162 | val rows: 1445


c:\Users\Fiona A\anaconda3\lib\site-packages\sklearn\preprocessing\_encoders.py:828: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Temporal MAE val (year=2024): $44,449


In [13]:
import pandas as pd

time_eval = X_time_val.copy()
time_eval["role_label_salary"] = X_time_val["role_label_salary"].values
time_eval["y_true"] = true_time_usd
time_eval["y_pred"] = pred_time_usd

role_mae_time = (
    time_eval.groupby("role_label_salary")[["y_true", "y_pred"]]
    .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
    .sort_values()
)

print("Temporal MAE by role (USD):")
display(role_mae_time.round(0))


Temporal MAE by role (USD):


role_label_salary
data_analyst                 33494.0
bi_engineer                  37497.0
other                        39989.0
project_manager              41910.0
data_engineer                42552.0
data_scientist               45968.0
machine_learning_engineer    48749.0
research_scientist           75054.0
dtype: float64

In [14]:
df_no_pm = df[df["role_label_salary"] != "project_manager"].copy()

X2_no_pm = df_no_pm[FEATURES_CATEGORICAL_NO_SRC + FEATURES_NUMERIC].copy()
y2_no_pm = df_no_pm["log_salary"].copy()

last_year = df_no_pm["work_year"].max()
train_mask = df_no_pm["work_year"] < last_year
val_mask = df_no_pm["work_year"] == last_year

Xtr = X2_no_pm[train_mask]; ytr = y2_no_pm[train_mask]
Xva = X2_no_pm[val_mask];   yva = y2_no_pm[val_mask]

pipe2_time_no_pm = Pipeline([("preprocess", pre2), ("model", model2)])
pipe2_time_no_pm.fit(Xtr, ytr)

pred = np.expm1(pipe2_time_no_pm.predict(Xva))
true = np.expm1(yva)

print(f"Temporal MAE (no project_manager) year={last_year}: ${mean_absolute_error(true, pred):,.0f}")


c:\Users\Fiona A\anaconda3\lib\site-packages\sklearn\preprocessing\_encoders.py:828: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Temporal MAE (no project_manager) year=2024: $44,503


In [15]:
import joblib
from pathlib import Path

pipe_final = Pipeline([("preprocess", pre2), ("model", model2)])
pipe_final.fit(X2, y2)

# Raíz del repo (la TechCareer interna)
PROJECT_ROOT = Path.cwd().parents[1]
print("PROJECT_ROOT:", PROJECT_ROOT)

MODEL_DIR = PROJECT_ROOT / "ml" / "models" / "salaries"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "v3_salary_from_profile_hgb.pkl"

# Guardamos el pipeline completo (preprocess + model)
joblib.dump(pipe_final, MODEL_PATH)

print("✅ Modelo guardado en:", MODEL_PATH)
print("¿Existe el fichero?", MODEL_PATH.exists())


c:\Users\Fiona A\anaconda3\lib\site-packages\sklearn\preprocessing\_encoders.py:828: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


PROJECT_ROOT: c:\Users\Fiona A\Desktop\IAPython\proyectos\TechCareer\TechCareer\backend
✅ Modelo guardado en: c:\Users\Fiona A\Desktop\IAPython\proyectos\TechCareer\TechCareer\backend\ml\models\salaries\v3_salary_from_profile_hgb.pkl
¿Existe el fichero? True


In [1]:
# Si tens pipe_final en memòria
print(pipe_final.named_steps["preprocess"].feature_names_in_)


NameError: name 'pipe_final' is not defined